# PRISM extension pipeline — v2 (preflight + T4 fixes)  — one run for all three NeurIPS 2026 submissions
**Use THIS file** (v2). Tells: has a **§0.5 Preflight** cell, §2 installs `sentencepiece`+`apt-get update`, §5 uses `_mdirs` guard, §7 sweeps `--model carbon`. 33 cells.


Produces every real number + figure for **ICBINB** (failure modes), **SIMBIOCHEM** (Proto/structure), and **MoML** (inverse design).
Run top-to-bottom on a **GPU** runtime. Primary results flow from the human-gene panel built in §3–4; the bacterial cross-validation (§6) is *optional*. Real-data-only sections (§10–12) need no GPU.

| Paper | Sections that feed it |
|---|---|
| ICBINB | §5 (collapse/artifact), §7 (decoding), §10 (audit + hub), §11 LOSO negative, §12 (fill paper) |
| SIMBIOCHEM | §8 (structural drift), §9 (Proto), §11 (recommender) |
| MoML | §11 (inverse design + design rules) |


## 0 · Mount Drive + unpack code


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/saanvi_prism.zip" "/content/"
!unzip -q -o "/content/saanvi_prism.zip" -d "/content/"
%cd /content/saanvi


## 0.5 · Preflight — verify the unpacked scripts are current
Catches a **stale zip** immediately (e.g. `structural_drift.py` missing `--max_genes`) instead of failing deep in the run. If this prints STALE, re-upload `saanvi_prism.zip` and re-run §0.


In [ ]:
# Robust: include only models whose outputs exist. Carbon vs GENERator alone
# already gives the collapse/artifact result; Evo2 is added if it generated.
_mdirs = {k:v for k,v in {'carbon':CARBON,'generator':GENERATOR,'evo2':EVO2}.items() if _has_gen(v)}
print('stress test on models:', list(_mdirs) or 'NONE (no outputs found!)')
_args = ' '.join(f'--model_dir {k}:{v}' for k,v in _mdirs.items())
get_ipython().system(f'python stress_test_generation.py {_args} --reference_fasta {PROMPTS} '
                     f'--prefix_frac 0.2 --cosmic_dir {COSMIC} --target_signatures SBS7a,SBS7b,SBS7c,SBS7d '
                     f'--output_dir {RES}/stress_test')
def _show(p):
    from IPython.display import Image, display
    import os
    display(Image(p)) if os.path.exists(p) else print('(no figure yet:', p, ')')
for p in ['A_collapse.png','D_stability.png']: _show(f'{RES}/stress_test/'+p)


## 1 · Config — edit paths once


In [ ]:
import os
BASE      = '/content/drive/MyDrive/PRISM/genmodel_bias'
PROMPTS   = f'{BASE}/prompts.fasta'   # existing 26-gene panel (carbon_output/generator_output came from THIS)
COSMIC    = '/content/drive/MyDrive/PRISM/cosmic_signatures'
PROFILES  = '/content/saanvi/profiles'
CENTROIDS = '/content/drive/MyDrive/PRISM/centroid_chunks'  # on your Drive (not in the zip)
RUNS      = '/content/saanvi/all_runs'
CARBON    = f'{BASE}/carbon_output'
GENERATOR = f'{BASE}/generator_output'
EVO2      = f'{BASE}/evo2_output'
RES       = f'{BASE}/results'
os.makedirs(RES, exist_ok=True); print('config set')


## 2 · Install dependencies


In [ ]:
!pip install -q biopython transformers accelerate torch statsmodels scipy pandas matplotlib scikit-learn joblib
!pip install -q sentencepiece tiktoken   # evo2 / HF tokenizers need these
!pip install -q --force-reinstall pyhmmer
!pip install -q flash-attn==2.8.0.post2 --no-build-isolation || echo 'flash-attn skipped (Evo2 native may not load on this GPU)'
!pip install -q evo2 || echo 'evo2 install failed -> Evo2 sections will skip'
!pip install -q git+https://github.com/evo-design/proto-tools.git || echo 'proto-tools optional'
!pip install -q git+https://github.com/evo-design/proto-language.git || echo 'proto-language optional'
!apt-get update -qq   # FIX: refresh index first, else blast/mafft 404
!apt-get install -y -qq ncbi-blast+ mafft > /dev/null 2>&1 && echo 'blast+mafft OK' || echo 'blast/mafft install failed (bacterial §6 will skip)'
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print('\u26a0 GPU compute capability', torch.cuda.get_device_capability(0), '< 8.0 (e.g. T4): Evo2 may not run. Carbon/GENERator/ESMFold are fine.')


## 3 · Build the human-gene prompt FASTA (primary panel)


In [ ]:
import os
from pathlib import Path
from Bio.Seq import Seq; from Bio.SeqRecord import SeqRecord; from Bio import SeqIO
if os.path.exists(PROMPTS):
    n = sum(1 for _ in SeqIO.parse(PROMPTS, 'fasta'))
    print(f'\u2713 reusing existing {PROMPTS} ({n} genes) \u2014 matches existing carbon_output/generator_output')
else:
    PROTEINS_DIR = '/content/drive/MyDrive/PRISM/proteins'  # <-- only used if prompts.fasta is absent
    recs, skip = [], []
    for txt in sorted(Path(PROTEINS_DIR).rglob('*.txt')):
        s = ''.join(txt.read_text().strip().upper().split())
        if not s or not set(s) <= set('ACGTN'): skip.append(str(txt)); continue
        recs.append(SeqRecord(Seq(s), id=txt.parent.name, description=f'source:{txt.name}'))
    SeqIO.write(recs, PROMPTS, 'fasta'); print(f'built {len(recs)} prompts, skipped {len(skip)}')


## 4 · Generate continuations — Carbon, GENERator, Evo 2 (human panel)
Evo 2 (single-nt tokenizer) is the control against the two 6-mer models.


In [ ]:
# Idempotent generation: skip any model whose continuations already exist.
# Set CARBON/GENERATOR/EVO2 in §1 to your EXISTING output dirs so already-run
# models are reused. Set FORCE_REGEN=True only if you changed generation params.
import glob, os
FORCE_REGEN = False
def _has_gen(d): return bool(glob.glob(os.path.join(d, '*_generated.fasta')))
def _has_ref(d): return bool(glob.glob(os.path.join(d, '*_expected_continuation.fasta')))
def _gen(cmd, outdir, label):
    if _has_gen(outdir) and not FORCE_REGEN:
        msg = f'\u2713 {label}: reusing outputs in {outdir} \u2014 skipping generation'
        if not _has_ref(outdir):
            msg += ('  \u26a0 WARNING: no *_expected_continuation.fasta here \u2014 these look like '
                    'whole-CDS-prompt outputs; the reference-based analysis (\u00a75/7/8) needs '
                    'prefix-prompt outputs (prefix_frac<1). Set FORCE_REGEN=True to regenerate.')
        print(msg)
    else:
        get_ipython().system(cmd)


In [ ]:
_gen(f"python generate_batch_carbon.py --input_fasta {PROMPTS} --model_name HuggingFaceBio/Carbon-500M "
     f"--num_samples 50 --batch_size 5 --max_new_tokens 500 --temperature 1.0 --top_k 4 "
     f"--strip_dna_tag --prefix_frac 0.2 --output_dir {CARBON}", CARBON, 'Carbon')


In [ ]:
_gen(f"python generate_batch_generator.py --input_fasta {PROMPTS} "
     f"--model_name GenerTeam/GENERator-v2-prokaryote-1.2b-base "
     f"--num_samples 50 --batch_size 5 --max_new_tokens 500 --temperature 1.0 --top_k 4 "
     f"--prefix_frac 0.2 --output_dir {GENERATOR}", GENERATOR, 'GENERator')


In [ ]:
_gen(f"python generate_batch_evo2.py --input_fasta {PROMPTS} --model_name evo2_7b "
     f"--num_samples 50 --batch_size 5 --max_new_tokens 500 --temperature 1.0 --top_k 4 "
     f"--prefix_frac 0.2 --output_dir {EVO2}", EVO2, 'Evo 2')


## 5 · [ICBINB] Cross-architecture failure-mode battery (human panel)
Collapse profile (A), tokenizer×alignment artifact isolation (B), bootstrap stability (D).


In [ ]:
# Robust: include only models whose outputs exist. Carbon vs GENERator alone
# already gives the collapse/artifact result; Evo2 is added if it generated.
_mdirs = {k:v for k,v in {'carbon':CARBON,'generator':GENERATOR,'evo2':EVO2}.items() if _has_gen(v)}
print('stress test on models:', list(_mdirs) or 'NONE (no outputs found!)')
_args = ' '.join(f'--model_dir {k}:{v}' for k,v in _mdirs.items())
get_ipython().system(f'python stress_test_generation.py {_args} --reference_fasta {PROMPTS} '
                     f'--prefix_frac 0.2 --cosmic_dir {COSMIC} --target_signatures SBS7a,SBS7b,SBS7c,SBS7d '
                     f'--output_dir {RES}/stress_test')
def _show(p):
    from IPython.display import Image, display
    import os
    display(Image(p)) if os.path.exists(p) else print('(no figure yet:', p, ')')
for p in ['A_collapse.png','D_stability.png']: _show(f'{RES}/stress_test/'+p)


## 6 · [ICBINB, OPTIONAL] Bacterial ortholog matched-alignment (MAFFT vs pairwise)
Secondary cross-validation. Needs a bacterial reference FASTA + bacterial generation outputs. Skip unless you have the 10-gene E. coli panel; the primary ICBINB results do **not** depend on this.


In [ ]:
# OPTIONAL bacterial cross-validation. Your real dirs use the _pilot suffix (10 genes).
BACT_REF   = f'{BASE}/bacterial_prompts.fasta'
BACT_CARB  = f'{BASE}/bacterial_carbon_output_pilot'      # existing (10 genes)
BACT_GEN   = f'{BASE}/bacterial_generator_output_pilot'   # existing (10 genes)
BACT_EVO2  = f'{BASE}/bacterial_evo2_output'              # generated below if missing
if os.path.exists(BACT_REF):
    _gen(f"python generate_batch_evo2.py --input_fasta {BACT_REF} --model_name evo2_7b "
         f"--num_samples 50 --batch_size 5 --max_new_tokens 500 --temperature 1.0 --top_k 4 "
         f"--prefix_frac 0.2 --output_dir {BACT_EVO2}", BACT_EVO2, 'Evo 2 (bacterial)')
    dirs = ' '.join(f'--model_dir {lbl}:{d}' for lbl,d in
                    [('carbon',BACT_CARB),('generator',BACT_GEN),('evo2',BACT_EVO2)] if _has_gen(d))
    get_ipython().system(f'python analyze_bacterial_generation_matched.py --bacterial_reference_fasta {BACT_REF} '
                         f'{dirs} --output_dir {RES}/bacterial_matched --prefix_frac 0.2')
else:
    print('bacterial_prompts.fasta not found -> skipping optional \u00a76 (primary results unaffected)')


## 7 · [ICBINB] Decoding-hyperparameter robustness (Experiment C, human panel)
Does the signal survive the temperature/top_k sweep? A signal that flips is a failure mode.


In [ ]:
# Decoding-robustness sweep. Uses Carbon-500M (fast + reliable on a T4);
# swap --model to evo2/generator on a bigger GPU. Pilot: 6 genes, T in {0.7,1.0,1.3}, k=4.
from Bio import SeqIO
SWEEP_PROMPTS = '/content/sweep_prompts.fasta'
SeqIO.write(list(SeqIO.parse(PROMPTS,'fasta'))[:6], SWEEP_PROMPTS, 'fasta')
!python sweep_decoding.py generate --model carbon --input_fasta {SWEEP_PROMPTS} \
    --temperatures 0.7,1.0,1.3 --top_ks 4 --num_samples 15 --prefix_frac 0.2 --sweep_dir {RES}/sweep
!python sweep_decoding.py analyze --sweep_dir {RES}/sweep --reference_fasta {SWEEP_PROMPTS} --prefix_frac 0.2 \
    --cosmic_dir {COSMIC} --target_signatures SBS7a,SBS7b,SBS7c,SBS7d --output_dir {RES}/sweep_analysis
def _show(p):
    from IPython.display import Image, display
    import os
    display(Image(p)) if os.path.exists(p) else print('(no figure yet:', p, ')')
_show(f'{RES}/sweep_analysis/sweep_robustness.png')


## 8 · [SIMBIOCHEM] Structural / energetic drift under signatures (ESMFold)
Turns embedding drift into structure+stability drift under a physical mutational process.


In [ ]:
# ESMFold is slow on a T4 -> pilot: 6 genes x 3 signatures x 5 draws, HF backend.
!python structural_drift.py --reference_fasta {PROMPTS} --profile_dir {PROFILES} \
    --signatures SBS17a,SBS1,SBS7a --n_draws 3 --max_genes 4 --max_aa 350 --folder hf \
    --output_dir {RES}/structural_drift
import os, pandas as pd
p = f'{RES}/structural_drift/structural_drift_summary.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('(structural_drift_summary.csv not written)')


## 9 · [SIMBIOCHEM] PRISM as a Proto directed-design campaign
Drives a wild-type toward PF17041 using the COSMIC signature as a physics-grounded generator; standalone MCMC fallback if Proto absent.


In [ ]:
# needs a seed CDS fasta (any single gene) + the centroid library
!python prism_proto_design.py --backend auto --wt_fasta {RUNS}/CL0072/pairs_cds.fasta \
    --target_pfam PF17041 --centroid_chunks_dir {CENTROIDS} \
    --signature_profile {PROFILES}/SBS-MS/SBS17a_PROFILE.txt \
    --generator signature --steps 300 --output_dir {RES}/prism_design_sbs17a
import pandas as pd; pd.read_csv(f'{RES}/prism_design_sbs17a/trajectory.csv').tail()


## 10 · [ICBINB, no GPU] Self-audit + hub-confound (existing outputs)
Corrected Table 1 (de-dup + self-pairs) and the hub-corrected steering figure.


In [ ]:
!python audit_prism_results.py --runs_dir {RUNS} --target_domain PF17041 --focus_signature SBS17a
!python analyze_steering_specificity.py --runs_dir {RUNS} --focus_signature SBS17a \
    --output_dir {RUNS}/steering_specificity
def _show(p):
    from IPython.display import Image, display
    import os
    display(Image(p)) if os.path.exists(p) else print('(no figure yet:', p, ')')
_show(f'{RUNS}/sbs17a_pf17041_ranking.png')
_show(f'{RUNS}/steering_specificity/steering_specificity.png')


## 11 · [MoML + ICBINB, no GPU] Inverse design + its failure mode
Multiclass recommender + design rules (MoML positive), and the Leave-Signatures-Out negative (ICBINB: mechanism-memorization confound).


In [ ]:
# positive result (works over the known 78-signature catalogue) + design_rules.csv
!python inverse_design_classifier.py --runs_dir {RUNS} --centroid_chunks_dir {CENTROIDS} \
    --profile_dir {PROFILES} --mode both --output_dir {RES}/inverse_design
# failure mode: generalization to UNSEEN signatures (leave-signatures-out)
!python inverse_design_classifier.py --runs_dir {RUNS} --centroid_chunks_dir {CENTROIDS} \
    --profile_dir {PROFILES} --mode loso --loso_folds 5 --output_dir {RES}/inverse_design_loso
import pandas as pd, json
dr = pd.read_csv(f'{RES}/inverse_design/design_rules.csv')
print('PF17041 design rule:'); display(dr[dr.target=='PF17041'].head())
print(json.load(open(f'{RES}/inverse_design_loso/inverse_design_metrics.json')).get('loso'))


## 12 · [ICBINB] Auto-fill the paper numbers + figures
Populates the red \pending markers and drops the real figures into the LaTeX.


In [ ]:
!python fill_paper_numbers.py --stress_dir {RES}/stress_test \
    --sweep_dir {RES}/sweep_analysis --paper_dir /content/saanvi --out /content/saanvi/paper_numbers.tex
print('paper_numbers.tex + fig_collapse.png/fig_sweep.png ready; rebuild icbinb_prism.tex')


## Outputs
- `results/stress_test/` — A/B/D CSVs + A_collapse.png (ICBINB II)
- `results/sweep_analysis/` — sweep_results.csv, sweep_robustness.png (ICBINB III)
- `results/structural_drift/` — structural_drift_summary.csv (SIMBIOCHEM)
- `results/prism_design_sbs17a/` — trajectory.csv, best_design.fasta (SIMBIOCHEM)
- `results/inverse_design/` — design_rules.csv, inverse_design_metrics.json (incl. LOSO), models (MoML + ICBINB)
- `all_runs/` — results_clean.csv, phase1_audit_summary.csv, sbs17a_pf17041_ranking.png, steering_specificity/ (ICBINB)
- `paper_numbers.tex`, `fig_collapse.png`, `fig_sweep.png` — auto-fill for icbinb_prism.tex
